# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Machine Learning: Logistic Regression** </center>
---
**Profesor**: Pablo Camarillo Ramirez

# Create SparkSession

In [10]:
from pcamarillor.spark_utils import SparkUtils
su = SparkUtils("ML: Logistic Regression", 
                "spark://spark-master:7077")
su.spark

# Collect Data

In [11]:
from pcamarillor.spark_utils import SparkUtils
# Create a small dataset as a list of tuples
# Format: (label, feature_x1, feature_x2)
data = [
    (1.0, 2.0, 3.0),
    (0.0, 1.0, 2.5),
    (1.0, 3.0, 5.0),
    (0.0, 0.5, 1.0),
    (1.0, 4.0, 6.0)
]

# Define schema for the DataFrame
schema = SparkUtils.generate_schema([("label", "float"), 
                                     ("feature_x1", "float"),
                                     ("feature_x2", "float")])

# Convert list to a DataFrame
df = su.spark.createDataFrame(data, schema)

### Assemble the features into a single vector column

In [12]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=["feature_x1", "feature_x2"], outputCol="features")
data_with_features = assembler.transform(df).select("label", "features")
data_with_features.show()
data_with_features.printSchema()                                   

+-----+---------+
|label| features|
+-----+---------+
|  1.0|[2.0,3.0]|
|  0.0|[1.0,2.5]|
|  1.0|[3.0,5.0]|
|  0.0|[0.5,1.0]|
|  1.0|[4.0,6.0]|
+-----+---------+

root
 |-- label: float (nullable = true)
 |-- features: vector (nullable = true)



# Data splitting
#### 80% training data and 20% testing data

In [13]:
train_df, test_df = data_with_features.randomSplit([0.8, 0.2], seed=57)

### Show dataset (for debugging)

In [14]:
print("Original Dataset")
df.show()

# Print train dataset
print("train set")
train_df.show()

Original Dataset
+-----+----------+----------+
|label|feature_x1|feature_x2|
+-----+----------+----------+
|  1.0|       2.0|       3.0|
|  0.0|       1.0|       2.5|
|  1.0|       3.0|       5.0|
|  0.0|       0.5|       1.0|
|  1.0|       4.0|       6.0|
+-----+----------+----------+

train set
+-----+---------+
|label| features|
+-----+---------+
|  0.0|[1.0,2.5]|
|  1.0|[2.0,3.0]|
|  0.0|[0.5,1.0]|
|  1.0|[4.0,6.0]|
+-----+---------+



# Create ML Model

In [15]:
from pyspark.ml.classification import LogisticRegression
lr = LogisticRegression(maxIter=10, regParam=0.01)

# Train ML Model

In [16]:
lr_model = lr.fit(train_df)

# Print coefficients
print("Coefficients: " + str(lr_model.coefficients))

# Display model summary
training_summary = lr_model.summary

Coefficients: [2.346116998875653,0.7963873036415707]


## Predictions

In [17]:
# Use the trained model to make predictions on the test data
predictions = lr_model.transform(test_df)

# Show predictions
predictions.select("features", "prediction", "probability").show()

+---------+----------+--------------------+
| features|prediction|         probability|
+---------+----------+--------------------+
|[3.0,5.0]|       1.0|[0.00524886113385...|
+---------+----------+--------------------+



# Test ML Model

In [18]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(labelCol="label",
                            predictionCol="prediction")

accuracy = evaluator.evaluate(predictions, 
                  {evaluator.metricName: "accuracy"})
print(f"Accuracy: {accuracy}")
precision = evaluator.evaluate(predictions,
                  {evaluator.metricName: "weightedPrecision"})
print(f"Precision: {precision}")
recall = evaluator.evaluate(predictions,
                  {evaluator.metricName: "weightedRecall"})
print(f"Recall: {recall}")
f1 = evaluator.evaluate(predictions,
                {evaluator.metricName: "f1"})
print(f"F1 Score: {f1}")  

Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0


# Lab 10: Logistic regression to predict heart disease

# Data collection

In [ ]:
# Define schema for the DataFrame
heart_schema = SparkUtils.generate_schema([
    ("male", "int"), 
    ("age", "int"), 
    ("education", "int"), 
    ("currentSmoker", "int"), 
    ("cigsPerDay", "int"), 
    ("BPMeds", "int"), 
    ("prevalentStroke", "int"), 
    ("prevalentHyp", "int"), 
    ("diabetes", "int"), 
    ("totChol", "int"), 
    ("sysBP", "float"), 
    ("diaBP", "float"), 
    ("BMI", "float"), 
    ("heartRate", "int"), 
    ("glucose", "int"), 
    ("TenYearCHD", "int")])

# Source: https://www.kaggle.com/datasets/dileep070/heart-disease-prediction-using-logistic-regression?resource=download

heart_df = su.spark.read \
                .option("header", "true") \
                .schema(heart_schema) \
                .csv("/opt/spark/work-dir/data/ml/logistic_regression/framingham.csv")

heart_df.printSchema()

26/04/16 03:19:17 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /opt/spark/work-dir/data/ml/logistic_regression/framingham.csv.
java.io.FileNotFoundException: File /opt/spark/work-dir/data/ml/logistic_regression/framingham.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:56)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:381)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/opt/spark/work-dir/data/ml/logistic_regression/framingham.csv. SQLSTATE: 42K03

26/04/16 03:41:50 ERROR TaskSchedulerImpl: Lost executor 1 on 172.18.0.8: worker lost: Not receiving heartbeat for 60 seconds
26/04/16 04:45:49 ERROR TaskSchedulerImpl: Lost executor 2 on 172.18.0.8: worker lost: Not receiving heartbeat for 60 seconds
26/04/16 04:47:48 ERROR TaskSchedulerImpl: Lost executor 3 on 172.18.0.8: worker lost: Not receiving heartbeat for 60 seconds
26/04/16 05:03:58 ERROR TaskSchedulerImpl: Lost executor 4 on 172.18.0.8: worker lost: Not receiving heartbeat for 60 seconds
26/04/16 05:51:39 ERROR TaskSchedulerImpl: Lost executor 5 on 172.18.0.8: worker lost: Not receiving heartbeat for 60 seconds
26/04/16 06:07:40 ERROR TaskSchedulerImpl: Lost executor 6 on 172.18.0.8: worker lost: Not receiving heartbeat for 60 seconds
26/04/16 06:52:30 ERROR TaskSchedulerImpl: Lost executor 7 on 172.18.0.8: worker lost: Not receiving heartbeat for 60 seconds
26/04/16 07:08:30 ERROR TaskSchedulerImpl: Lost executor 8 on 172.18.0.8: worker lost: Not receiving heartbeat for 60 

In [ ]:
heart_df.show(2)

# Data Splitting

# Create ML Model

# Train ML Model

# Test ML Model

In [ ]:
su.spark.stop()